## Check if there are 0 in NPi colums

In [1]:
# imports
import pandas as pd

In [2]:
df = pd.read_excel("../Pupilometri/NPi_Light_off_redcapdata.xlsx")

cols = ["npi_right", "npi_left", "npi_right_2", "npi_left_2"]

# Convert everything to numeric
for col in cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Combine first day, and all follow-ups
df["npi_right_final"] = df["npi_right"].combine_first(df["npi_right_2"])
df["npi_left_final"]  = df["npi_left"].combine_first(df["npi_left_2"])



# Rows where either eye NPi is 0
zero_rows = df[
    (df["npi_right_final"] == 0) |
    (df["npi_left_final"] == 0)
]

# print(f"Rows with zero in active measurement: {len(zero_rows)}")
# print(zero_rows) # COMMENTED FOR PRIVACY

In [3]:
for idx, row in zero_rows.iterrows():
    excel_row = idx + 2  # adjust for the numbering in Excel
    #print(f"Row {excel_row}, record_id: {row['record_id']}")

In [4]:
missing_rows = df[
    df["npi_right_final"].isna() |
    df["npi_left_final"].isna()
]

print(f"Rows with missing value in at least one eye: {len(missing_rows)}")

Rows with missing value in at least one eye: 17


In [5]:
for idx, row in missing_rows.iterrows():
    excel_row = idx + 2
    
    right_missing = pd.isna(row["npi_right_final"])
    left_missing  = pd.isna(row["npi_left_final"])
    
    # print(
    #     f"Row {excel_row}, record_id: {row['record_id']}, "
    #     f"right_missing: {right_missing}, left_missing: {left_missing}"
    # )

## Now check dates if they are correct order:

In [6]:
# Convert to datetime
df["date_1"] = pd.to_datetime(df["date_examination"], errors="coerce")
df["date_2"] = pd.to_datetime(df["date_of_examination_2"], errors="coerce")

# Combine into one column
df["date_final"] = df["date_1"].combine_first(df["date_2"])

In [7]:
df["prev_date"] = df.groupby("record_id")["date_final"].shift(1)

# Simple check if a date happens before it should
wrong_order = df[
    df["date_final"] < df["prev_date"]
]

print(f"Rows with incorrect date order: {len(wrong_order)}")

Rows with incorrect date order: 14


In [8]:
for idx, row in wrong_order.iterrows():
    excel_row = idx + 2
    # print(
    #     f"Row {excel_row}, record_id: {row['record_id']}, "
    #     f"date: {row['date_final']}, previous: {row['prev_date']}"
    # )